## Setup and Data Loading
As before, we'll import the packages we'll need in this notebook. Most of these are the same as the previous notebook, but there are a few new ones.

In [1]:
import os
import matplotlib
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sklearn.model_selection
import torch
import torch.nn as nn
import torch.optim as optim
import torchinfo
import torchvision
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from torch.utils.data import DataLoader, random_split, Subset
from sklearn.model_selection import train_test_split
from torchinfo import summary
from torchvision import datasets, transforms
from tqdm import tqdm
import shutil
from collections import Counter
from pathlib import Path
import re
from collections import defaultdict
from PIL import Image

import torch
from torch.utils.data import Dataset
from torchvision import transforms
import torch.nn.utils.rnn as rnn_utils

In [2]:
print("torch version : ", torch.__version__)
print("torchvision version : ", torchvision.__version__)
print("torchinfo version : ", torchinfo.__version__)
print("numpy version : ", np.__version__)
print("matplotlib version : ", matplotlib.__version__)

!python --version

torch version :  2.6.0+cu124
torchvision version :  0.21.0+cu124
torchinfo version :  1.8.0
numpy version :  1.26.4
matplotlib version :  3.7.2
Python 3.11.13


In [ ]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Using {device} device.")

## Data Preparation

In [ ]:
# Define source and destination paths
source_base = "/kaggle/input/ucf-crime-dataset/Train"
destination_base = "/kaggle/working/dataset/Train"

# List of directories to copy
directories_to_copy = [
    "Robbery",
    "Shooting", 
    "Stealing",
    "Burglary",
    "NormalVideos"
]

### for testing code
# directories_to_copy = [
#     "Robbery",
#     "Shooting"
    
# ]

def copy_crime_dataset():
    """
    Copy specified crime dataset directories from input to working directory
    """
    try:
        # Create the destination base directory if it doesn't exist
        os.makedirs(destination_base, exist_ok=True)
        print(f"Created destination directory: {destination_base}")
        
        # Copy each specified directory
        for directory in directories_to_copy:
            source_path = os.path.join(source_base, directory)
            destination_path = os.path.join(destination_base, directory)
            
            # Check if source directory exists
            if os.path.exists(source_path):
                # Copy the entire directory tree
                shutil.copytree(source_path, destination_path, dirs_exist_ok=True)
                
                # Count files in the copied directory
                file_count = len([f for f in Path(destination_path).rglob('*') if f.is_file()])
                print(f"✓ Copied {directory}: {file_count} files")
            else:
                print(f"✗ Source directory not found: {source_path}")
    
    except Exception as e:
        print(f"Error occurred: {str(e)}")
        return False
    
    return True

def verify_copy():
    """
    Verify that the directories were copied successfully
    """
    print("\n--- Verification ---")
    for directory in directories_to_copy:
        dest_path = os.path.join(destination_base, directory)
        if os.path.exists(dest_path):
            file_count = len([f for f in Path(dest_path).rglob('*') if f.is_file()])
            print(f"✓ {directory}: {file_count} files present")
        else:
            print(f"✗ {directory}: Directory not found")

if __name__ == "__main__":
    print("Starting UCF Crime Dataset copy process...")
    print(f"Source: {source_base}")
    print(f"Destination: {destination_base}")
    print("-" * 50)
    
    success = copy_crime_dataset()
    
    if success:
        verify_copy()
        print("\n✓ Copy process completed!")
    else:
        print("\n✗ Copy process failed!")

In [ ]:
# # copying 10 files for testing purpose
# def copy_camnuvem_data(src_dir="/kaggle/input/camnuvem/images/training/anomaly", dst_dir="/kaggle/working/dataset/Train/Robbery"):
#     # Make sure destination exists
#     os.makedirs(dst_dir, exist_ok=True)
    
#     # Counter to limit to 10 files
#     file_count = 0
#     max_files = 10
    
#     # Loop through each subfolder inside src_dir
#     for subfolder in os.listdir(src_dir):
#         subfolder_path = os.path.join(src_dir, subfolder)
        
#         if os.path.isdir(subfolder_path):  # ensure it's a folder
#             for filename in os.listdir(subfolder_path):
#                 if file_count >= max_files:  # Stop after copying 10 files
#                     break
                
#                 src_file = os.path.join(subfolder_path, filename)
                
#                 # Only process files, not directories
#                 if os.path.isfile(src_file):
#                     # New filename = subfolder prefix + "_" + original filename
#                     new_filename = f"{subfolder}_{filename}"
#                     dst_file = os.path.join(dst_dir, new_filename)
                    
#                     shutil.copy(src_file, dst_file)
#                     file_count += 1
#                     print(f"Copied: {src_file} -> {dst_file}")
    
#     print(f"✅ {file_count} files copied and renamed successfully from {src_dir} to {dst_dir}!")

In [ ]:
# copying the whole files
def copy_camnuvem_data(src_dir = "/kaggle/input/camnuvem/images/training/anomaly", dst_dir = "/kaggle/working/dataset/Train/Robbery" ):
    
    # Make sure destination exists
    os.makedirs(dst_dir, exist_ok=True)
    
    # Loop through each subfolder inside src_dir
    for subfolder in os.listdir(src_dir):
        subfolder_path = os.path.join(src_dir, subfolder)
        
        if os.path.isdir(subfolder_path):  # ensure it's a folder
            for filename in os.listdir(subfolder_path):
                src_file = os.path.join(subfolder_path, filename)
                
                # New filename = subfolder prefix + "_" + original filename
                new_filename = f"{subfolder}_{filename}"
                dst_file = os.path.join(dst_dir, new_filename)
                
                shutil.copy(src_file, dst_file)
                
    
    print(f"✅ Files copied and renamed successfully from {src_dir} to {dst_dir}!")


In [ ]:
# copy anomaly images
copy_camnuvem_data()

#copy normal images
copy_camnuvem_data("/kaggle/input/camnuvem/images/training/normal", "/kaggle/working/dataset/Train/NormalVideos")

## Load Data

In [ ]:
data_dir = "/kaggle/working/dataset/Train"

print("Data directory:", data_dir)

We'll be applying the same transformations we have been all along:


* Resize images to 64 x 64
* Convert images to PyTorch tensors
* Normalize the tensors

In [ ]:
transform = transforms.Compose(
    [
        transforms.Resize((64, 64)),
        transforms.ToTensor()
    ]
)
transform

In [ ]:
dataset = datasets.ImageFolder(root= data_dir, transform=transform)
dataset

In [ ]:
batch_size = 32
dataset_loader = DataLoader(dataset, batch_size=batch_size)

# Get one batch
first_batch = next(iter(dataset_loader))

print(f"Shape of one batch: {first_batch[0].shape}")
print(f"Shape of labels: {first_batch[1].shape}")

In [ ]:
classes = dataset.classes
classes

In [ ]:
def get_mean_std(loader):
    """Computes the mean and standard deviation of image data.

    Input: a `DataLoader` producing tensors of shape [batch_size, channels, pixels_x, pixels_y]
    Output: the mean of each channel as a tensor, the standard deviation of each channel as a tensor
            formatted as a tuple (means[channels], std[channels])"""

    channels_sum, channels_squared_sum, num_batches = 0, 0, 0
    for data, _ in tqdm(loader, desc="Computing mean and std", leave=False):
        channels_sum += torch.mean(data, dim=[0, 2, 3])
        channels_squared_sum += torch.mean(data**2, dim=[0, 2, 3])
        num_batches += 1
    mean = channels_sum / num_batches
    std = (channels_squared_sum / num_batches - mean**2) ** 0.5

    return mean, std

In [ ]:
mean, std = get_mean_std(dataset_loader)

print(f"Mean: {mean}")
print(f"Standard deviation: {std}")

In [ ]:
# normalize it 
transform_norm = transforms.Compose(
    [
        transforms.Resize((64, 64)),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std)
    ]
)

## Load Video Frame

set the max_frames to 20, you can change the number to any of the true length

In [ ]:
import re

fnamesq = "437_frame_18"
video_id = re.match(r"(.*?(?:_|-).*?)(?:_|-)", fnamesq).group(1)

print(video_id)  # Output: Normal_Videos001


In [ ]:
class VideoFrameDataset(Dataset):
    def __init__(self, root_dir, transform=None, max_frames=20):
        """
        Args:
            root_dir (str): Root directory with subfolders (one per class).
            transform (callable): Transform to apply to each frame.
            max_frames (int): Number of frames to pad/truncate videos to.
        """
        self.root_dir = root_dir
        self.transform = transform
        self.max_frames = max_frames

        self.samples = []  # (video_frames, label)

        # Map classes to numeric labels
        self.class_to_idx = {
            cls_name: idx
            for idx, cls_name in enumerate(sorted(os.listdir(root_dir)))
        }

        # Walk through each class folder
        for cls_name, label in self.class_to_idx.items():
            class_dir = os.path.join(root_dir, cls_name)
            groups = self._group_frames_by_video(class_dir)

            for frames in groups.values():
                self.samples.append((frames, label))

    def _group_frames_by_video(self, folder):
        groups = defaultdict(list)
        for fname in sorted(os.listdir(folder)):
            if fname.lower().endswith((".png", ".jpg", ".jpeg")):
                video_id = re.match(r"(.*?(?:_|-).*?)(?:_|-)", fname).group(1)
                groups[video_id].append(os.path.join(folder, fname))
        return groups

    def _load_video_frames(self, frame_paths):
        frames = []
        for f in frame_paths:
            img = Image.open(f).convert("RGB")
            if self.transform:
                img = self.transform(img)
            else:
                img = transforms.ToTensor()(img)
            frames.append(img)

        frames = torch.stack(frames, dim=0)  # [num_frames, C, H, W]
        return frames

    def _pad_or_truncate(self, frames):
        num_frames, C, H, W = frames.shape
        mask = torch.ones(self.max_frames, dtype=torch.bool)  # 1 = real frame

        if num_frames < self.max_frames:
            pad = torch.zeros(self.max_frames - num_frames, C, H, W)
            frames = torch.cat([frames, pad], dim=0)
            mask[num_frames:] = 0  # mark padding as 0
        else:
            frames = frames[:self.max_frames]

        return frames, mask, num_frames  # return real length too

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        frame_paths, label = self.samples[idx]
        frames = self._load_video_frames(frame_paths)
        frames, mask, true_len = self._pad_or_truncate(frames)
        return frames, mask, true_len, label


In [ ]:
# load dataset
dataset_norm = VideoFrameDataset(
    root_dir=data_dir,   
    transform=transform_norm,
    max_frames=20)

### Training and Validation Split

In [ ]:
# ---- 2. Extract labels ----
labels = [dataset_norm[i][3] for i in range(len(dataset_norm))]
labels

In [ ]:
unique_labels = list(set(labels))
unique_labels

In [ ]:
indices = list(range(len(dataset_norm)))
indices

In [ ]:
train_idx, val_idx = train_test_split(
    indices,
    test_size=0.2,       # 20% validation
    random_state=42
    stratify = unique_labels
)

train_dataset = Subset(dataset_norm, train_idx)
val_dataset   = Subset(dataset_norm, val_idx)


In [ ]:
g = torch.Generator()
g.manual_seed(42)


batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

val_loader = DataLoader(val_dataset, batch_size=batch_size)

single_batch = next(iter(train_loader))[0]
print(f"Shape of one batch: {single_batch.shape}")

In [ ]:
frames, masks, labels, class_idx = next(iter(train_loader))
print(f"Frames: {frames.shape}")      # (B, T, C, H, W)
print(f"Masks: {masks.shape}")        # (B, T)
print(f"Labels: {labels.shape}")      # (B,)
print(f"Class idx: {class_idx.shape}")# (B,)


In [ ]:
# for training set
for videos, masks, lengths, labels in train_loader:
    print("Videos:", videos.shape)   # [B, max_frames, C, H, W]
    print("Masks:", masks.shape)     # [B, max_frames]
    print("true Lengths:", lengths)       # [B]
    print("Labels:", labels)         # [B]
    break


In [ ]:
def class_counts(dataset):
    c = Counter(x[3] for x in tqdm(dataset))
    class_to_index = dataset.dataset.class_to_idx
    return pd.Series({cat: c[idx] for cat, idx in class_to_index.items()})

In [ ]:
train_class_distributions = class_counts(train_dataset)

train_class_distributions

In [ ]:
# Create a bar plot from train_class_distribution
train_class_distributions.sort_values().plot(kind="bar")

# Add axis labels and title
plt.xlabel("Class Label")
plt.ylabel("Frequency [count]")
plt.title("Class Distribution in Training Set");

In [ ]:
# Get the class distribution
validation_class_distributions = class_counts(val_dataset)

# Create a bar plot from train_class_distribution
validation_class_distributions.sort_values().plot(kind="bar")

# Add axis labels and title
plt.xlabel("Class Label")
plt.ylabel("Frequency [count]")
plt.title("Class Distribution in Validation Set");

## Model Arc

### Implementing Transfer Learning

Will use resnet for transfer learning

In [ ]:
class VideoClassifier(nn.Module):
    def __init__(self, num_classes=5, lstm_hidden_size=512, lstm_num_layers=1, pretrained=False):
        super(VideoClassifier, self).__init__()
        
        # Load ResNet50 as backbone (pretrained=False for offline compatibility)
        self.backbone = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.DEFAULT if pretrained else None)
        
        # Freeze the backbone parameters
        for param in self.backbone.parameters():
            param.requires_grad = False
        
        # Replace FC with Identity to output features (2048-dim)
        self.backbone.fc = nn.Identity()
        
        # LSTM layer
        self.lstm = nn.LSTM(input_size=2048, hidden_size=lstm_hidden_size, num_layers=lstm_num_layers, batch_first=True)
        
        # Classifier after LSTM
        self.classifier = nn.Sequential(
            nn.Linear(lstm_hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.5),  # Explicit dropout rate
            nn.Linear(256, num_classes)
        )

    def forward(self, videos, lengths):
        B, T, C, H, W = videos.shape
        
        # Reshape to process each frame through the backbone
        videos_reshaped = videos.view(B * T, C, H, W)  # [B*T, C, H, W]
        features = self.backbone(videos_reshaped)  # [B*T, 2048]
        features = features.view(B, T, 2048)  # [B, T, 2048]
        
        # Pack sequences to handle variable lengths
        lengths_cpu = lengths.cpu()  # PyTorch requires lengths on CPU
        packed_features = rnn_utils.pack_padded_sequence(features, lengths_cpu, batch_first=True, enforce_sorted=False)
        
        # Pass through LSTM
        packed_output, (h, c) = self.lstm(packed_features)
        
        # Take the last hidden state
        out = h[-1]  # [B, lstm_hidden_size]
        
        # Pass through classifier
        out = self.classifier(out)  # [B, num_classes]
        return out


In [ ]:
# Create the model 
model = VideoClassifier(num_classes=5, pretrained=False)  # change num_classes to number of class say 4 or 4

# Move model to device
model.to(device)

# Test with a batch from train_loader
test_videos, test_masks, test_lengths, test_labels = next(iter(train_loader))
test_videos = test_videos.to(device)
test_masks = test_masks.to(device)  # Not used in forward but moved for completeness
test_lengths = test_lengths.to(device)
test_labels = test_labels.to(device)


In [ ]:
print("\nTest batch shapes:")
print(f"Videos: {test_videos.shape}")  # [B, T, C, H, W]
print(f"Masks: {test_masks.shape}")   # [B, T]
print(f"Lengths: {test_lengths.shape}")  # [B]
print(f"Labels: {test_labels.shape}")   # [B]



# Function to reset the classifier parameters
def reset_classifier(model):
    model.classifier[0].reset_parameters()  # First Linear
    model.classifier[3].reset_parameters()  # Second Linear
    print("Classifier parameters reset.")

# Call reset if needed
reset_classifier(model)

# 
optimizer = torch.optim.Adam(model.classifier.parameters(), lr=0.001)  
loss_fn = torch.nn.CrossEntropyLoss()


## Training model 

In [ ]:

# def train_epoch(model, optimizer, loss_fn, data_loader, device="cpu"):
#     # INSERT ...
#     # REMOVE{
#     training_loss = 0.0
#     model.train()

#     # Iterate over all batches in the training set to complete one epoch
#     for inputs, targets in tqdm(data_loader, desc="Training", leave=False):
#         optimizer.zero_grad()
#         inputs = inputs.to(device)
#         targets = targets.to(device)

#         output = model(inputs)
#         loss = loss_fn(output, targets)

#         loss.backward()
#         optimizer.step()
#         training_loss += loss.data.item() * inputs.size(0)

#     return training_loss / len(data_loader.dataset)
#     # REMOVE}


# def predict(model, data_loader, device="cpu"):
#     # INSERT ...
#     # REMOVE{
#     all_probs = torch.tensor([]).to(device)

#     model.eval()
#     with torch.no_grad():
#         for inputs, targets in tqdm(data_loader, desc="Predicting", leave=False):
#             inputs = inputs.to(device)
#             output = model(inputs)
#             probs = torch.nn.functional.softmax(output, dim=1)
#             all_probs = torch.cat((all_probs, probs), dim=0)

#     return all_probs
#     # REMOVE}


# def score(model, data_loader, loss_fn, device="cpu"):
#     # INSERT ...
#     # REMOVE{
#     total_loss = 0
#     total_correct = 0

#     model.eval()
#     with torch.no_grad():
#         for inputs, targets in tqdm(data_loader, desc="Scoring", leave=False):
#             inputs = inputs.to(device)
#             output = model(inputs)

#             targets = targets.to(device)
#             loss = loss_fn(output, targets)
#             total_loss += loss.data.item() * inputs.size(0)

#             correct = torch.eq(torch.argmax(output, dim=1), targets)
#             total_correct += torch.sum(correct).item()
#     average_loss = total_loss / len(data_loader.dataset)
#     accuracy = total_correct / len(data_loader.dataset)
#     return average_loss, accuracy
#     # REMOVE}


# def train(model, optimizer, loss_fn, train_loader, val_loader, epochs=20, device="cpu"):
#     # INSERT ...
#     # REMOVE{
#     for epoch in range(1, epochs + 1):
#         # Train one epoch
#         training_loss = train_epoch(model, optimizer, loss_fn, train_loader, device)

#         # Test on validation set
#         validation_loss, validation_accuracy = score(model, val_loader, loss_fn, device)

#         print(
#             f"Epoch: {epoch}, Training Loss: {training_loss:.2f}, "
#             f"Validation Loss: {validation_loss:.2f}, Validation accuracy = {validation_accuracy:.2f}"
#         )



In [ ]:
def train_epoch(model, optimizer, loss_fn, data_loader, device="cpu"):
    training_loss = 0.0
    model.train()

    # Iterate over all batches in the training set to complete one epoch
    for videos, masks, lengths, labels in tqdm(data_loader, desc="Training", leave=False):
        videos = videos.to(device)
        lengths = lengths.to(device)
        labels = labels.to(device)
        # Note: masks are not used in the model but are included in the loader

        optimizer.zero_grad()
        output = model(videos, lengths)  # Pass videos and lengths to the model
        loss = loss_fn(output, labels)

        loss.backward()
        optimizer.step()
        training_loss += loss.data.item() * videos.size(0)

    return training_loss / len(data_loader.dataset)

def predict(model, data_loader, device="cpu"):
    all_probs = torch.tensor([]).to(device)

    model.eval()
    with torch.no_grad():
        for videos, masks, lengths, labels in tqdm(data_loader, desc="Predicting", leave=False):
            videos = videos.to(device)
            lengths = lengths.to(device)
            # Note: masks and labels are not used in the model for prediction

            output = model(videos, lengths)  # Pass videos and lengths to the model
            probs = torch.nn.functional.softmax(output, dim=1)
            all_probs = torch.cat((all_probs, probs), dim=0)

    return all_probs

def score(model, data_loader, loss_fn, device="cpu"):
    total_loss = 0
    total_correct = 0

    model.eval()
    with torch.no_grad():
        for videos, masks, lengths, labels in tqdm(data_loader, desc="Scoring", leave=False):
            videos = videos.to(device)
            lengths = lengths.to(device)
            labels = labels.to(device)
            # Note: masks are not used in the model

            output = model(videos, lengths)  # Pass videos and lengths to the model
            loss = loss_fn(output, labels)
            total_loss += loss.data.item() * videos.size(0)

            correct = torch.eq(torch.argmax(output, dim=1), labels)
            total_correct += torch.sum(correct).item()
    
    average_loss = total_loss / len(data_loader.dataset)
    accuracy = total_correct / len(data_loader.dataset)
    return average_loss, accuracy

def train(model, optimizer, loss_fn, train_loader, val_loader, epochs=20, device="cpu"):
    for epoch in range(1, epochs + 1):
        # Train one epoch
        training_loss = train_epoch(model, optimizer, loss_fn, train_loader, device)

        # Test on validation set
        validation_loss, validation_accuracy = score(model, val_loader, loss_fn, device)

        print(
            f"Epoch: {epoch}, Training Loss: {training_loss:.2f}, "
            f"Validation Loss: {validation_loss:.2f}, Validation accuracy = {validation_accuracy:.2f}"
        )

### Training

In [ ]:
#train model with number of model
train(
    model,
    optimizer,
    loss_fn,
    train_loader,
    val_loader,
    epochs=5,
    device=device,
)

In [ ]:
# prediction on the val set
probabilities = predict(model, val_loader, device)
predictions = torch.argmax(probabilities, dim = 1)

plot confusion matrix

In [ ]:
targets = []

for _, _, _, labels in tqdm(val_loader):
    targets.extend(labels.tolist())

In [ ]:
Classes = dataset.classes

In [ ]:
cm = confusion_matrix(targets, predictions.cpu())

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=Classes)

disp.plot(cmap=plt.cm.Blues, xticks_rotation="vertical")
plt.show();